## Setup

Connect to Elasticsearch and load both models: `BGEM3FlagModel` to embed the query (dense + sparse), and `FlagReranker` to rerank the retrieved candidates.

In [1]:
import json
from pathlib import Path

import torch
from elasticsearch import Elasticsearch
from FlagEmbedding import BGEM3FlagModel, FlagReranker

ES_HOST = "http://localhost:9200"
INDEX_NAME = "document-chunks"

EMBED_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

es = Elasticsearch(ES_HOST)
info = es.info()
print(f"Connected to {info['cluster_name']} (Elasticsearch {info['version']['number']})")

embed_model = BGEM3FlagModel(EMBED_MODEL_NAME, use_fp16=(DEVICE == "cuda"), device=DEVICE)
reranker = FlagReranker(RERANKER_MODEL_NAME, use_fp16=(DEVICE == "cuda"), device=DEVICE)

d:\Code\Learning\ElasticSearch\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Connected to docker-cluster (Elasticsearch 8.15.0)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1851.48it/s]


## Embed the query

Same bge-m3 call as indexing (Step 3), just on a single query string instead of 247 chunks. The sparse token names get the same `.` -> `·` sanitization applied at indexing time (Step 3/4) -- otherwise a query token like `"3.00"` couldn't match the corresponding indexed field name.

In [2]:
def embed_query(query_text):
    output = embed_model.encode(
        [query_text],
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
    )
    dense_vec = [float(x) for x in output["dense_vecs"][0]]

    # convert_id_to_token collapses its result to a bare dict (not a 1-item list)
    # whenever the input batch has exactly one entry -- which is always true here
    # since we always encode a single query string.
    raw_sparse = embed_model.convert_id_to_token(output["lexical_weights"])
    if isinstance(raw_sparse, list):
        raw_sparse = raw_sparse[0]
    sparse_weights = {tok.replace(".", "\u00b7"): float(w) for tok, w in raw_sparse.items()}

    return dense_vec, sparse_weights

## Hybrid retrieval: dense kNN + sparse `rank_feature`, fused with RRF

Two independent candidate lists, fused client-side:

- **Dense**: `knn` search on the `embedding` field (cosine similarity over bge-m3's dense vectors).
- **Sparse**: a `bool`/`should` of `rank_feature` clauses, one per query token, boosted by that token's bge-m3 weight -- this is lexical matching using bge-m3's *learned* term weights instead of raw BM25 statistics. Only the top ~32 highest-weighted query tokens are used, to keep the query small; a token that never appears in any indexed chunk simply contributes zero hits, it does not error (verified against the live index).
- **Fusion**: Reciprocal Rank Fusion (`score = sum(1 / (k + rank))` across lists, `k=60` -- the standard default from the original RRF paper). RRF combines *ranks*, not raw scores, which sidesteps the problem that cosine similarity (0-1) and `rank_feature` scores (unbounded, saturation-based) live on incomparable scales -- summing raw scores would let whichever signal happens to have larger numbers dominate.

In [3]:
SOURCE_EXCLUDES = {"excludes": ["embedding", "sparse_embedding"]}


def knn_search(query_vector, k=20, num_candidates=100):
    resp = es.search(
        index=INDEX_NAME,
        knn={"field": "embedding", "query_vector": query_vector, "k": k, "num_candidates": num_candidates},
        source=SOURCE_EXCLUDES,
    )
    return resp["hits"]["hits"]


def sparse_search(sparse_weights, size=20, top_n_tokens=32):
    top_tokens = sorted(sparse_weights.items(), key=lambda kv: -kv[1])[:top_n_tokens]
    should_clauses = [
        {"rank_feature": {"field": f"sparse_embedding.{token}", "boost": weight}}
        for token, weight in top_tokens
    ]
    resp = es.search(
        index=INDEX_NAME,
        query={"bool": {"should": should_clauses}},
        size=size,
        source=SOURCE_EXCLUDES,
    )
    return resp["hits"]["hits"]


def reciprocal_rank_fusion(result_lists, k=60):
    fused = {}
    for hits in result_lists:
        for rank, hit in enumerate(hits, start=1):
            entry = fused.setdefault(hit["_id"], {"rrf_score": 0.0, "hit": hit})
            entry["rrf_score"] += 1.0 / (k + rank)
    return sorted(fused.values(), key=lambda e: -e["rrf_score"])


def hybrid_retrieve(query_text, dense_k=20, sparse_k=20, fused_top_n=20):
    dense_vec, sparse_weights = embed_query(query_text)

    dense_hits = knn_search(dense_vec, k=dense_k, num_candidates=dense_k * 5)
    sparse_hits = sparse_search(sparse_weights, size=sparse_k)

    fused = reciprocal_rank_fusion([dense_hits, sparse_hits])
    return fused[:fused_top_n]

### Sanity check: fused retrieval

Run a demo query through `hybrid_retrieve` and inspect the fused ranking before any reranking happens.

In [4]:
demo_query = "What does the green light symbolize?"
fused_candidates = hybrid_retrieve(demo_query)

print(f"{len(fused_candidates)} fused candidates for: {demo_query!r}\n")
for i, c in enumerate(fused_candidates[:10], start=1):
    src = c["hit"]["_source"]
    print(f"{i:>2}. rrf={c['rrf_score']:.4f}  {src['chunk_id']}  (ch {src['chapter_title']})")
    print(f"      {src['text'][:150]}...")

20 fused candidates for: 'What does the green light symbolize?'

 1. rrf=0.0328  great-gatsby-ch09-s05-c002  (ch IX)
      And as I sat there brooding on the old, unknown world, I thought of Gatsby’s wonder when he first picked out the green light at the end of Daisy’s doc...
 2. rrf=0.0320  great-gatsby-ch05-s01-c000  (ch V)
      After the house, we were to see the grounds and the swimming pool, and the hydroplane, and the midsummer flowers—but outside Gatsby’s window it began ...
 3. rrf=0.0315  great-gatsby-ch09-s05-c001  (ch IX)
      On the last night, with my trunk packed and my car sold to the grocer, I went over and looked at that huge incoherent failure of a house once more. On...
 4. rrf=0.0281  great-gatsby-ch03-s00-c002  (ch III)
      By seven o’clock the orchestra has arrived, no thin five-piece affair, but a whole pitful of oboes and trombones and saxophones and viols and cornets ...
 5. rrf=0.0275  great-gatsby-ch03-s00-c003  (ch III)
      The lights grow brighter as 

## Rerank with bge-reranker-v2-m3

The fused list is retrieval-quality ranking (fast, approximate). The cross-encoder reads each `(query, chunk_text)` pair *jointly* and outputs a precise relevance score -- much slower per pair, which is exactly why it only runs on the ~20 already-fused candidates instead of the whole index. `normalize=True` applies a sigmoid so scores land in `[0, 1]` instead of raw logits.

In [4]:
def rerank(query_text, candidates, top_n=5):
    pairs = [[query_text, c["hit"]["_source"]["text"]] for c in candidates]
    scores = reranker.compute_score(pairs, normalize=True, max_length=1024)

    for c, score in zip(candidates, scores):
        c["rerank_score"] = score

    return sorted(candidates, key=lambda c: -c["rerank_score"])[:top_n]

### Sanity check: reranking

Rerank the same fused candidates and compare the final order (and scores) against the RRF-only ranking above -- expect some reshuffling, since RRF only knows about rank position while the cross-encoder actually reads the text against the query.

In [6]:
reranked = rerank(demo_query, fused_candidates, top_n=5)

print(f"Top {len(reranked)} after reranking:\n")
for i, c in enumerate(reranked, start=1):
    src = c["hit"]["_source"]
    print(f"{i}. rerank={c['rerank_score']:.4f}  (rrf={c['rrf_score']:.4f})  {src['chunk_id']}  (ch {src['chapter_title']})")
    print(f"   {src['text'][:200]}...")
    print()

Top 5 after reranking:

1. rerank=0.2763  (rrf=0.0328)  great-gatsby-ch09-s05-c002  (ch IX)
   And as I sat there brooding on the old, unknown world, I thought of Gatsby’s wonder when he first picked out the green light at the end of Daisy’s dock. He had come a long way to this blue lawn, and h...

2. rerank=0.2259  (rrf=0.0315)  great-gatsby-ch09-s05-c001  (ch IX)
   On the last night, with my trunk packed and my car sold to the grocer, I went over and looked at that huge incoherent failure of a house once more. On the white steps an obscene word, scrawled by some...

3. rerank=0.1574  (rrf=0.0320)  great-gatsby-ch05-s01-c000  (ch V)
   After the house, we were to see the grounds and the swimming pool, and the hydroplane, and the midsummer flowers—but outside Gatsby’s window it began to rain again, so we stood in a row looking at the...

4. rerank=0.0794  (rrf=0.0143)  great-gatsby-ch01-s02-c004  (ch I)
   Already it was deep summer on roadhouse roofs and in front of wayside garages, 

## End-to-end search function

Wraps embedding -> hybrid retrieval -> RRF fusion -> reranking into one call.

In [9]:
def search(query_text, dense_k=20, sparse_k=20, fused_top_n=20, final_top_n=5):
    fused = hybrid_retrieve(query_text, dense_k=dense_k, sparse_k=sparse_k, fused_top_n=fused_top_n)
    return rerank(query_text, fused, top_n=final_top_n)


def print_results(query_text, results):
    print(f"Query: {query_text!r}\n")
    for i, c in enumerate(results, start=1):
        src = c["hit"]["_source"]
        print(f"{i}. [{c['rerank_score']:.4f}] {src['chunk_id']}  (Chapter {src['chapter_title']})")
        print(f"   {src['text'][:200]}...")
        print()


for query_text in [
    "What does the green light symbolize?",
    "Describe Gatsby\'s parties",
    "How does Tom Buchanan treat Myrtle Wilson?",
]:
    print_results(query_text, search(query_text))
    print("=" * 80)

Query: 'What does the green light symbolize?'

1. [0.2763] great-gatsby-ch09-s05-c002  (Chapter IX)
   And as I sat there brooding on the old, unknown world, I thought of Gatsby’s wonder when he first picked out the green light at the end of Daisy’s dock. He had come a long way to this blue lawn, and h...

2. [0.2259] great-gatsby-ch09-s05-c001  (Chapter IX)
   On the last night, with my trunk packed and my car sold to the grocer, I went over and looked at that huge incoherent failure of a house once more. On the white steps an obscene word, scrawled by some...

3. [0.1574] great-gatsby-ch05-s01-c000  (Chapter V)
   After the house, we were to see the grounds and the swimming pool, and the hydroplane, and the midsummer flowers—but outside Gatsby’s window it began to rain again, so we stood in a row looking at the...

4. [0.0794] great-gatsby-ch01-s02-c004  (Chapter I)
   Already it was deep summer on roadhouse roofs and in front of wayside garages, where new red petrol-pumps sat out in

### Gemini

In [5]:
import os
from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-2.5-flash"  # nhanh + rẻ, đủ tốt cho RAG Q&A; đổi sang gemini-2.5-pro nếu cần chất lượng cao hơn

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise RuntimeError(
        "Chưa có GOOGLE_API_KEY trong biến môi trường. "
        "Lấy API key tại https://aistudio.google.com/apikey rồi set biến môi trường trước khi mở Jupyter."
    )

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

### Ghép context từ top-N kết quả

Mỗi đoạn được đánh số và gắn nhãn chương/`chunk_id`, để Gemini có thể trích dẫn nguồn cụ thể trong câu trả lời thay vì trả lời chung chung không rõ lấy từ đâu.

In [6]:
def build_context(results):
    blocks = []
    for i, c in enumerate(results, start=1):
        src = c["hit"]["_source"]
        blocks.append(
            f"[Đoạn {i} - Chapter {src['chapter_title']}, chunk_id={src['chunk_id']}]\n{src['text']}"
        )
    return "\n\n".join(blocks)

### Prompt & gọi Gemini

`SYSTEM_PROMPT` là phần quan trọng nhất để chống hallucination -- ép model chỉ dùng context được cung cấp, không tự bịa thêm kiến thức ngoài sách, và phải nói rõ khi context không đủ thay vì đoán bừa. `temperature=0.2` giữ câu trả lời bám sát context, hạn chế model "sáng tạo" thêm chi tiết không có thật.

In [10]:
SYSTEM_PROMPT = """Bạn là trợ lý trả lời câu hỏi về tiểu thuyết "The Great Gatsby" của F. Scott Fitzgerald.
Chỉ được trả lời dựa trên các đoạn trích (context) được cung cấp bên dưới -- KHÔNG dùng kiến thức bên ngoài, KHÔNG suy đoán hay bịa thêm chi tiết không có trong context.
Nếu context không đủ thông tin để trả lời, hãy nói rõ là không tìm thấy thông tin liên quan trong các đoạn được cung cấp, đừng cố trả lời.
Khi trả lời, trích dẫn chương liên quan (ví dụ: "(Chapter III)") cho các chi tiết quan trọng."""


def generate_answer(query_text, results, model=GEMINI_MODEL):
    context = build_context(results)
    user_prompt = f"""Context:
{context}

Câu hỏi: {query_text}

Trả lời:"""

    response = gemini_client.models.generate_content(
        model=model,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.2,
        ),
    )
    return response.text

## End-to-end: `ask()`

Gộp toàn bộ pipeline: `search()` (embed -> hybrid retrieve -> RRF -> rerank) -> `build_context()` -> `generate_answer()`.

In [19]:
def ask(query_text, final_top_n=5):
    results = search(query_text, final_top_n=final_top_n)
    answer = generate_answer(query_text, results)
    return answer, results


demo_answer, demo_results = ask("Where does the story take place? Name the two primary fictional areas mentioned in Long Island.")

print("Answer:\n", demo_answer)
print("\nSources:")
for c in demo_results:
    src = c["hit"]["_source"]
    print(f"  - {src['chunk_id']} (Chapter {src['chapter_title']}, rerank_score={c['rerank_score']:.4f})")

Answer:
 Câu chuyện diễn ra trên một hòn đảo ở phía đông New York, cách thành phố hai mươi dặm, nơi có hai khối đất hình quả trứng khổng lồ nhô ra Vịnh Long Island (Long Island Sound) (Chapter I, Chapter V).

Hai khu vực hư cấu chính được đề cập là:
1.  **West Egg**: Nơi người kể chuyện và Gatsby sinh sống. Nó được mô tả là "ít thời thượng hơn trong hai khu vực" (Chapter I).
2.  Context đề cập đến "một cặp trứng khổng lồ" và West Egg là "ít thời thượng hơn trong hai khu vực", nhưng không nêu tên cụ thể của khu vực còn lại (Chapter I).

Sources:
  - great-gatsby-ch01-s01-c005 (Chapter I, rerank_score=0.2580)
  - great-gatsby-ch02-s00-c015 (Chapter II, rerank_score=0.1284)
  - great-gatsby-ch06-s01-c008 (Chapter VI, rerank_score=0.1233)
  - great-gatsby-ch06-s00-c001 (Chapter VI, rerank_score=0.1050)
  - great-gatsby-ch01-s01-c004 (Chapter I, rerank_score=0.0475)
